# Waveform Uniformity Model

**CRISP-DM Phase 4 — Modeling**  
Predicting nozzle-to-nozzle variability (`sd_std`) from waveform parameters.  
Target: find which parameter combinations produce the most consistent print output across all nozzles.

See `documentation/model-1-within-chip-uniformity.md` for design decisions and trade-offs.

In [ ]:
import os, sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_absolute_error

# XGBoost requires libomp on macOS — install with: brew install libomp
try:
    from xgboost import XGBRegressor
    XGBOOST_AVAILABLE = True
except Exception:
    XGBOOST_AVAILABLE = False
    print('XGBoost not available — skipping. Run: brew install libomp')

## 1. Load and aggregate
*CRISP-DM Phase 3 — Data Preparation*

The processed dataset has one row per printed sheet. Each waveform condition was tested on ~6 replicate sheets.  
We average `sd_std` over those replicates so each row represents the true uniformity of a waveform setting, not a single noisy measurement.

In [ ]:
df = pd.read_parquet('../../data/processed/waveform_tuning_row_summary.parquet')

condition_cols = ['Color$', 'HeadIdx#', 'V', 'F_r', 'dt2', 'Coverage#']
agg = df.groupby(condition_cols)[['sd_std', 'sd_mean']].mean().reset_index()
agg = agg.rename(columns={'sd_std': 'sd_std_mean', 'sd_mean': 'sd_mean_mean'})

print(f'Raw rows:          {len(df):,}')
print(f'After aggregation: {len(agg):,} conditions')

## 2. Feature engineering
*CRISP-DM Phase 3 — Data Preparation*

Raw waveform parameters are extended with interaction and polynomial terms to capture non-linear relationships observed in the coverage analysis.

In [ ]:
color_dummies = pd.get_dummies(agg['Color$'], prefix='color', drop_first=False)

base = agg[['V', 'F_r', 'dt2', 'Coverage#']].copy()

# Interaction and polynomial terms motivated by the coverage analysis:
# V and F_r are physically coupled; dt2 interacts with coverage; both V and F_r show non-linear effects
base['V_x_Fr']         = agg['V'] * agg['F_r']
base['dt2_x_coverage'] = agg['dt2'] * agg['Coverage#']
base['V_sq']           = agg['V'] ** 2
base['Fr_sq']          = agg['F_r'] ** 2

X = pd.concat([base, color_dummies], axis=1)
y = agg['sd_std_mean']

print(f'Features: {X.columns.tolist()}')
print(f'Target range: {y.min():.5f} – {y.max():.5f}')

## 3. Train / test split
*CRISP-DM Phase 4 — Modeling: Generate Test Design*

Split by `HeadIdx#`: train on chips 1–24, test on chips 25–30.  
This tests whether the model generalises to unseen physical printheads — the realistic scenario for deployment.

In [ ]:
train_mask = agg['HeadIdx#'] <= 24

X_train, X_test = X[train_mask], X[~train_mask]
y_train, y_test = y[train_mask], y[~train_mask]

print(f'Train: {len(X_train):,} rows ({train_mask.mean():.0%})')
print(f'Test:  {len(X_test):,} rows ({(~train_mask).mean():.0%})')

## 4. Train and evaluate models
*CRISP-DM Phase 4 — Modeling: Build Model*

Three model candidates are trained and compared. The best is selected automatically by R² on the test set.

In [ ]:
models = {
    'Linear Regression': LinearRegression(),
    'Random Forest':     RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1),
}
if XGBOOST_AVAILABLE:
    models['XGBoost'] = XGBRegressor(n_estimators=200, learning_rate=0.05, max_depth=5,
                                     random_state=42, verbosity=0)

results = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    results[name] = {
        'R2':    r2_score(y_test, y_pred),
        'MAE':   mean_absolute_error(y_test, y_pred),
        'model': model,
        'y_pred': y_pred,
    }

summary = pd.DataFrame({k: {'R²': v['R2'], 'MAE': v['MAE']} for k, v in results.items()}).T
print(summary.round(4))

## 5. Predicted vs actual
*CRISP-DM Phase 5 — Evaluation*

Visual check: points close to the diagonal mean the model predicts well. Scatter indicates where it struggles.

In [ ]:
n = len(results)
fig, axes = plt.subplots(1, n, figsize=(5 * n, 4))
if n == 1:
    axes = [axes]

for ax, (name, res) in zip(axes, results.items()):
    ax.scatter(y_test, res['y_pred'], alpha=0.3, s=5)
    lims = [y_test.min(), y_test.max()]
    ax.plot(lims, lims, 'r--', linewidth=1)
    ax.set_xlabel('Actual sd_std')
    ax.set_ylabel('Predicted sd_std')
    ax.set_title(f'{name}\nR²={res["R2"]:.3f}  MAE={res["MAE"]:.5f}')

plt.tight_layout()
plt.show()

## 6. Feature importance (Random Forest)
*CRISP-DM Phase 5 — Evaluation: Assess Model*

This is a **model validation step**, not a new analysis. It checks whether the model learned meaningful patterns from the data by comparing what it relies on against what the earlier coverage analysis already found.

### What the model found vs what the earlier analysis found

| Feature | Model importance | Earlier finding |
|---|---|---|
| Coverage# | **#1 (27%)** | Confirmed — sd_std increases 2–3× from low to high coverage |
| V / V² | #3–4 (combined ~22%) | Confirmed — V is the strongest *controllable* driver; V=30 is 2.2× worse than V=20 |
| Color | combined ~28% | Confirmed — coverage analysis already showed different sd_std ranges per color (Y: 1.7× factor, K: 3.0×) |
| F_r / F_r² | combined ~15% | Confirmed — F_r has a non-linear effect; optimal F_r for uniformity differs from optimal F_r for density |
| dt2 | 0.8% | Confirmed — dt2 has near-zero effect on uniformity |

### Key takeaway
The model's feature ranking is **fully consistent** with the earlier analysis — it learned the same patterns the data exploration already revealed.  
Coverage# dominates statistically, but it is fixed by the print job. Among the actually tunable parameters (V, F_r, dt2), the model agrees: **V dominates, dt2 barely matters**.

In [ ]:
rf = results['Random Forest']['model']
importance = pd.Series(rf.feature_importances_, index=X_train.columns).sort_values()

fig, ax = plt.subplots(figsize=(8, 5))
importance.plot(kind='barh', ax=ax, color='steelblue')
ax.set_title('Feature importance — Random Forest')
ax.set_xlabel('Importance score')
plt.tight_layout()
plt.show()

## 7. Optimal waveform settings
*CRISP-DM Phase 5 — Evaluation: Determine next steps*

Use the best model to predict `sd_std` across all tested parameter combinations.  
Results are grouped by Color and Coverage so you can find the best V, F_r, dt2 for any desired ink density level.

In [ ]:
best_name = max(results, key=lambda k: results[k]['R2'])
best_model = results[best_name]['model']
print(f'Best model: {best_name}  (R²={results[best_name]["R2"]:.4f})')

param_cols = ['Color$', 'V', 'F_r', 'dt2', 'Coverage#']
grid = agg[param_cols].drop_duplicates().reset_index(drop=True).copy()

color_dummies_grid = pd.get_dummies(grid['Color$'], prefix='color', drop_first=False)
base_grid = grid[['V', 'F_r', 'dt2', 'Coverage#']].copy()
base_grid['V_x_Fr']         = grid['V'].values * grid['F_r'].values
base_grid['dt2_x_coverage'] = grid['dt2'].values * grid['Coverage#'].values
base_grid['V_sq']           = grid['V'].values ** 2
base_grid['Fr_sq']          = grid['F_r'].values ** 2

X_grid = pd.concat([base_grid, color_dummies_grid], axis=1)
X_grid = X_grid.reindex(columns=X_train.columns, fill_value=0)

grid['predicted_sd_std'] = best_model.predict(X_grid)

# Best V, F_r, dt2 per (Color, Coverage) — coverage is fixed by the desired ink density
best_per_coverage = (
    grid.sort_values('predicted_sd_std')
    .groupby(['Color$', 'Coverage#'], group_keys=False)
    .head(1)
    .sort_values(['Color$', 'Coverage#'])
)
print()
print(best_per_coverage[['Color$', 'Coverage#', 'V', 'F_r', 'dt2', 'predicted_sd_std']].to_string(index=False))

## Save model

Save the trained model to disk so it can be loaded by the dashboard without retraining.

In [ ]:
import joblib

joblib.dump(best_model, '../../models/rf_sd_std.pkl')
joblib.dump(X_train.columns.tolist(), '../../models/rf_sd_std_features.pkl')
print(f'Model saved: {best_name}')
print(f'Features saved: {X_train.columns.tolist()}')

## 8. Manual prediction
*CRISP-DM Phase 5 — Evaluation*

Change the values below and run the cell to get a predicted `sd_std` for any waveform setting.

In [ ]:
# --- change these values ---
V        = 20.0   # options: 20, 22, 24, 26, 28, 30
F_r      = 1.26   # options: 1.02, 1.08, 1.09, ..., 1.379
dt2      = -1100  # options: -1100, -900, -700, -500, -300
coverage = 2.0    # options: 2 – 31
color    = 'M'    # options: C, M, Y, K
# ---------------------------

custom = pd.DataFrame([{'V': V, 'F_r': F_r, 'dt2': dt2, 'Coverage#': coverage, 'Color$': color}])

color_dummies = pd.get_dummies(custom['Color$'], prefix='color', drop_first=False)
base = custom[['V', 'F_r', 'dt2', 'Coverage#']].copy()
base['V_x_Fr']         = custom['V'] * custom['F_r']
base['dt2_x_coverage'] = custom['dt2'] * custom['Coverage#']
base['V_sq']           = custom['V'] ** 2
base['Fr_sq']          = custom['F_r'] ** 2

X_custom = pd.concat([base, color_dummies], axis=1)
X_custom = X_custom.reindex(columns=X_train.columns, fill_value=0)

pred = best_model.predict(X_custom)[0]

# Look up actual value if this exact combination exists in the data
match = agg[
    (agg['V'] == V) & (agg['F_r'] == F_r) & (agg['dt2'] == dt2) &
    (agg['Coverage#'] == coverage) & (agg['Color$'] == color)
]

print(f'Settings: V={V}, F_r={F_r}, dt2={dt2}, Coverage={coverage}, Color={color}')
print(f'Predicted sd_std: {pred:.6f}')
if len(match) > 0:
    actual = match['sd_std_mean'].values[0]
    print(f'Actual    sd_std: {actual:.6f}  (error: {abs(pred - actual):.6f})')
else:
    print('Actual: not in dataset — prediction only')
print(f'Lower is better. Best in dataset: {y_train.min():.6f}')

## 9. Find best settings for a target SD level
*CRISP-DM Phase 5 — Evaluation*

Instead of just minimising std globally, this cell finds settings that:  
1. Produce an sd_mean (ink density) close to your **target** — within a **tolerance** you define  
2. Among those, return the ones with the **lowest sd_std** (most uniform nozzles)  

This answers the real question: *which settings give me the right darkness AND the most consistent nozzles?*

In [ ]:
# --- change these values ---
target_sd_mean = 0.35   # desired ink density level
tolerance      = 0.02   # how close sd_mean must be to the target (±)
color          = 'C'    # options: C, M, Y, K
# ---------------------------

# Filter conditions within tolerance of target sd_mean for the chosen color
filtered = agg[
    (agg['Color$'] == color) &
    (agg['sd_mean_mean'] >= target_sd_mean - tolerance) &
    (agg['sd_mean_mean'] <= target_sd_mean + tolerance)
]

if len(filtered) == 0:
    print(f'No conditions found for Color={color} with sd_mean between {target_sd_mean - tolerance:.3f} and {target_sd_mean + tolerance:.3f}')
    print(f'sd_mean range for {color}: {agg[agg["Color$"]==color]["sd_mean_mean"].min():.3f} – {agg[agg["Color$"]==color]["sd_mean_mean"].max():.3f}')
else:
    best = (
        filtered.sort_values('sd_std_mean')
        .groupby(['V', 'F_r', 'dt2'], group_keys=False)
        .head(1)
        .sort_values('sd_std_mean')
        .head(10)
    )
    print(f'Color={color} | Target sd_mean={target_sd_mean} ± {tolerance} | {len(filtered)} matching conditions found')
    print()
    print(best[['Color$', 'V', 'F_r', 'dt2', 'Coverage#', 'sd_mean_mean', 'sd_std_mean']]
          .rename(columns={'sd_mean_mean': 'sd_mean', 'sd_std_mean': 'sd_std'})
          .to_string(index=False))